# Probabilities from Simulations and Sales Data (ABA Unit 2, Practical 2)

**Aim:** calculate probabilities of different outcomes from simulated experiments, and conditional probabilities and Bayes' theorem from the sales data.

Run the cells in order (Runtime → Run all). Nothing to install. The note `conditional-probability-practical` on Sutra explains every step.

In [1]:
# Step 1. Load the sales data and set up the random generator
import numpy as np
import pandas as pd

url = ("https://raw.githubusercontent.com/Prathameshv07/ai-ds-notes/main/"
       "content/assets/data/sahyadri-mart-sales.csv")
df = pd.read_csv(url, parse_dates=["date"])
rng = np.random.default_rng(2026)
print("Rows, columns:", df.shape)

Rows, columns: (8772, 16)


In [2]:
# Step 2. Simulate 100,000 throws of two dice and tabulate the sums
d1 = rng.integers(1, 7, size=100_000)
d2 = rng.integers(1, 7, size=100_000)
total = d1 + d2
sim = pd.Series(total).value_counts(normalize=True).sort_index()
exact = pd.Series({s: (6 - abs(s - 7)) / 36 for s in range(2, 13)})
print(pd.DataFrame({"simulated": sim.round(4), "exact": exact.round(4)}).to_string())

    simulated   exact
2      0.0280  0.0278
3      0.0560  0.0556
4      0.0842  0.0833
5      0.1106  0.1111
6      0.1394  0.1389
7      0.1670  0.1667
8      0.1383  0.1389
9      0.1109  0.1111
10     0.0832  0.0833
11     0.0550  0.0556
12     0.0274  0.0278


In [3]:
# Step 3. Probabilities of events: simulated vs exact
first_even = d1 % 2 == 0
print(f"P(sum >= 10)           : {(total >= 10).mean():.4f}   exact {6/36:.4f}")
print(f"P(doubles)             : {(d1 == d2).mean():.4f}   exact {6/36:.4f}")
print(f"P(sum is 8)            : {(total == 8).mean():.4f}   exact {5/36:.4f}")
print(f"P(sum is 8 | 1st even) : {(total[first_even] == 8).mean():.4f}   exact {3/18:.4f}")

P(sum >= 10)           : 0.1656   exact 0.1667
P(doubles)             : 0.1656   exact 0.1667
P(sum is 8)            : 0.1383   exact 0.1389
P(sum is 8 | 1st even) : 0.1655   exact 0.1667


In [4]:
# Step 4. Two events in the sales data: festival day, and high-sales day (a store's top 10%)
p90 = df.groupby("store_id")["revenue_inr"].transform(lambda s: s.quantile(0.9))
df["high"] = (df["revenue_inr"] > p90).astype(int)
print(f"P(festival) = {df['is_festival'].mean():.4f}")
print(f"P(high)     = {df['high'].mean():.4f}")

P(festival) = 0.0438
P(high)     = 0.0999


In [5]:
# Step 5. Joint counts, conditional probabilities and a test of independence
print(pd.crosstab(df["is_festival"], df["high"], margins=True).to_string())
p_high_fest = df.loc[df["is_festival"] == 1, "high"].mean()
p_high_not = df.loc[df["is_festival"] == 0, "high"].mean()
print(f"\nP(high | festival)     = {p_high_fest:.3f}")
print(f"P(high | not festival) = {p_high_not:.3f}")
both = (df["high"] & df["is_festival"]).mean()
print(f"P(high and festival) = {both:.4f}  vs  P(high) x P(festival) = {df['high'].mean() * df['is_festival'].mean():.4f}")

high            0    1   All
is_festival                 
0            7766  622  8388
1             130  254   384
All          7896  876  8772

P(high | festival)     = 0.661
P(high | not festival) = 0.074
P(high and festival) = 0.0290  vs  P(high) x P(festival) = 0.0044


In [6]:
# Step 6. Bayes' theorem: P(festival | high), by formula and by direct count
p_f, p_h = df["is_festival"].mean(), df["high"].mean()
bayes = p_high_fest * p_f / p_h
direct = df.loc[df["high"] == 1, "is_festival"].mean()
print(f"By Bayes' theorem: {bayes:.3f}   By direct count: {direct:.3f}")

By Bayes' theorem: 0.290   By direct count: 0.290
